# BrainTumor-Agent：BraTS2021 + nnU-Net V2（Google Colab GPU）

这个 Notebook 用于真实训练项目中的四模态 3D 分割模型。它会完成：BraTS2021 数据检查与转换、nnU-Net V2 规划和预处理、GPU 训练、断点续训、验证及模型导出。

> 默认使用 `demo_100epochs`：完整 BraTS2021 数据、fold 0、100 epochs，适合实习项目演示。它是实际训练，不是随机假结果，但不能等同于标准 1000 epochs × 5 folds 的完整实验。若要正式实验，将 `TRAINING_PROFILE` 改为 `full_1000epochs`，并依次训练 0～4 folds。

> 本项目仅用于医学影像辅助研究与教学，不用于独立诊断或临床决策。

## 0. 运行前准备

1. 在 Colab 菜单选择“代码执行程序 → 更改运行时类型 → T4 GPU（或其他 NVIDIA GPU）”。
2. 将 `BrainTumor-Agent-Colab.zip` 放到 Google Drive 的 `MyDrive/BrainTumor-Agent/`。
3. 从官方授权渠道取得 BraTS2021 训练集，将训练集压缩为 `BraTS2021_TrainingData.zip`，放到同一目录。压缩包解压后应包含病例目录，每个病例包含 `*_t1.nii.gz`、`*_t1ce.nii.gz`、`*_t2.nii.gz`、`*_flair.nii.gz`、`*_seg.nii.gz`。
4. 按顺序运行全部单元格。Colab 断线后重新运行；训练单元会自动发现 `checkpoint_latest.pth` 并续训。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ===== 只需按实际 Drive 位置修改这一区域 =====
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/BrainTumor-Agent')
PROJECT_ZIP = DRIVE_PROJECT_DIR / 'BrainTumor-Agent-Colab.zip'
BRATS_SOURCE = DRIVE_PROJECT_DIR / 'BraTS2021_TrainingData.zip'  # 也可以改成已解压目录

TRAINING_PROFILE = 'demo_100epochs'  # 可选：demo_100epochs / full_1000epochs
FOLD = '0'                          # 完整实验依次改为 0、1、2、3、4
DATASET_ID = 137
CONFIGURATION = '3d_fullres'
PLANNER = 'nnUNetPlannerResEncM'
PLANS = 'nnUNetResEncUNetMPlans'
PREPROCESS_PROCESSES = 2            # Colab 内存紧张时设为 1
DATA_AUGMENTATION_PROCESSES = 2     # T4/双核 Colab 建议 2

TRAINERS = {
    'demo_100epochs': 'nnUNetTrainer_100epochs',
    'full_1000epochs': 'nnUNetTrainer',
}
if TRAINING_PROFILE not in TRAINERS:
    raise ValueError(f'未知训练配置：{TRAINING_PROFILE}')
TRAINER = TRAINERS[TRAINING_PROFILE]

# 原始数据和 checkpoint 持久化到 Drive；预处理数组复制到 Colab 本地盘以提高训练速度。
PERSIST_ROOT = DRIVE_PROJECT_DIR / 'nnUNet_storage'
LOCAL_WORK_ROOT = Path('/content/brain_tumor_agent_work')
LOCAL_PROJECT_PARENT = LOCAL_WORK_ROOT / 'project'
LOCAL_BRATS_ROOT = LOCAL_WORK_ROOT / 'brats_source'
LOCAL_NNUNET_ROOT = LOCAL_WORK_ROOT / 'nnunet_workspace'

print('训练配置：', TRAINING_PROFILE)
print('Trainer：', TRAINER, 'Fold：', FOLD)
print('持久化目录：', PERSIST_ROOT)

## 1. 解压项目并安装训练依赖

Colab 自带 GPU 版 PyTorch，因此先保留运行时自带的 PyTorch，再安装 nnU-Net V2。若下面显示 PyTorch 2.9.x，会给出性能警告；nnU-Net 官方指出该版本的 3D 卷积 AMP 存在明显性能回退。

In [ ]:
import shutil
import sys

DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if not PROJECT_ZIP.is_file():
    print(f'Drive 中未找到 {PROJECT_ZIP.name}，请选择本机项目压缩包上传。')
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError('请只上传一个 BrainTumor-Agent-Colab.zip')
    PROJECT_ZIP.write_bytes(uploaded[zip_names[0]])

if LOCAL_PROJECT_PARENT.exists():
    shutil.rmtree(LOCAL_PROJECT_PARENT)
LOCAL_PROJECT_PARENT.mkdir(parents=True)
shutil.unpack_archive(str(PROJECT_ZIP), str(LOCAL_PROJECT_PARENT))

project_candidates = [
    path.parent.parent
    for path in LOCAL_PROJECT_PARENT.rglob('segmentation/train.py')
]
if len(project_candidates) != 1:
    raise RuntimeError('项目压缩包中应包含且仅包含一个 segmentation/train.py')
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))
print('项目目录：', PROJECT_ROOT)

In [ ]:
import importlib.metadata
import subprocess

torch_version_before = importlib.metadata.version('torch')
print('Colab 预装 PyTorch：', torch_version_before)
if tuple(int(part) for part in torch_version_before.split('+')[0].split('.')[:2]) >= (2, 9):
    print('警告：nnU-Net 官方报告 PyTorch 2.9 的 3D AMP 性能明显下降。可继续运行，但训练可能较慢。')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'nnunetv2>=2.5,<3.0', 'SimpleITK>=2.4,<3.0', 'nibabel>=5.3,<6.0'
], check=True)

import torch  # noqa: E402

if not torch.cuda.is_available():
    raise RuntimeError('未检测到 CUDA。请在 Colab 中选择 GPU 运行时，然后从头运行。')
if shutil.which('nnUNetv2_train') is None:
    raise RuntimeError('nnUNetv2_train 未安装成功')

print('PyTorch：', torch.__version__)
print('CUDA：', torch.version.cuda)
print('GPU：', torch.cuda.get_device_name(0))
print('显存(GB)：', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 2. 建立可断点续训的 nnU-Net 工作区

`nnUNet_raw` 和 `nnUNet_results` 位于 Drive，模型 checkpoint 会持续保存；`nnUNet_preprocessed` 位于 Colab 本地盘，并从 Drive 缓存恢复，以避免直接从 Drive 读取大量训练切片。

In [ ]:
import os

PERSIST_RAW = PERSIST_ROOT / 'nnUNet_raw'
PERSIST_PREPROCESSED = PERSIST_ROOT / 'nnUNet_preprocessed'
PERSIST_RESULTS = PERSIST_ROOT / 'nnUNet_results'
for path in (PERSIST_RAW, PERSIST_PREPROCESSED, PERSIST_RESULTS):
    path.mkdir(parents=True, exist_ok=True)

LOCAL_NNUNET_ROOT.mkdir(parents=True, exist_ok=True)
local_raw = LOCAL_NNUNET_ROOT / 'nnUNet_raw'
local_preprocessed = LOCAL_NNUNET_ROOT / 'nnUNet_preprocessed'
local_results = LOCAL_NNUNET_ROOT / 'nnUNet_results'

for link_path, target_path in ((local_raw, PERSIST_RAW), (local_results, PERSIST_RESULTS)):
    if link_path.is_symlink():
        link_path.unlink()
    elif link_path.exists():
        raise RuntimeError(f'拒绝覆盖非软链接路径：{link_path}')
    link_path.symlink_to(target_path, target_is_directory=True)

if local_preprocessed.exists():
    shutil.rmtree(local_preprocessed)
if any(PERSIST_PREPROCESSED.iterdir()):
    print('正在从 Drive 恢复预处理缓存到 Colab 本地盘……')
    shutil.copytree(PERSIST_PREPROCESSED, local_preprocessed)
else:
    local_preprocessed.mkdir(parents=True)

os.environ['nnUNet_raw'] = str(local_raw)
os.environ['nnUNet_preprocessed'] = str(local_preprocessed)
os.environ['nnUNet_results'] = str(local_results)
os.environ['nnUNet_n_proc_DA'] = str(DATA_AUGMENTATION_PROCESSES)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print('nnUNet_raw：', local_raw.resolve())
print('nnUNet_preprocessed：', local_preprocessed)
print('nnUNet_results：', local_results.resolve())

## 3. BraTS2021 转换、完整性检查和预处理

如果 Drive 中已经存在 Dataset137 的 raw 数据和完整预处理缓存，本单元会跳过重复转换。首次运行需要提供官方 BraTS2021 训练集，合成测试 NIfTI 不能用于训练。

In [ ]:
from segmentation.prepare_dataset import find_brats_case_dirs

dataset_raw_dir = PERSIST_RAW / 'Dataset137_BraTS2021'
plans_path = local_preprocessed / 'Dataset137_BraTS2021' / f'{PLANS}.json'
needs_prepare = not (dataset_raw_dir / 'dataset.json').is_file()
needs_preprocess = not plans_path.is_file()

BRATS_DIR = None
if needs_prepare:
    if not BRATS_SOURCE.exists():
        raise FileNotFoundError(
            f'未找到 BraTS 数据：{BRATS_SOURCE}。请先从官方授权渠道取得训练集并上传到 Drive。'
        )
    if LOCAL_BRATS_ROOT.exists():
        shutil.rmtree(LOCAL_BRATS_ROOT)
    if BRATS_SOURCE.is_file():
        LOCAL_BRATS_ROOT.mkdir(parents=True)
        print('正在从 Drive 解压 BraTS2021 到 Colab 本地盘……')
        shutil.unpack_archive(str(BRATS_SOURCE), str(LOCAL_BRATS_ROOT))
        search_root = LOCAL_BRATS_ROOT
    else:
        search_root = BRATS_SOURCE

    candidates = [search_root]
    candidates.extend(path for path in search_root.iterdir() if path.is_dir())
    best = None
    for candidate in candidates:
        try:
            case_dirs = find_brats_case_dirs(candidate)
        except Exception:
            continue
        if best is None or len(case_dirs) > best[0]:
            best = (len(case_dirs), candidate)
    if best is None:
        raise RuntimeError('压缩包中未发现合法 BraTS 病例目录')
    case_count, BRATS_DIR = best
    print(f'发现 {case_count} 个病例：{BRATS_DIR}')
else:
    print('nnUNet_raw/Dataset137_BraTS2021 已存在，跳过原始数据转换。')

In [ ]:
if needs_prepare:
    prepare_command = [
        sys.executable, '-m', 'segmentation.prepare_dataset',
        '--brats-dir', str(BRATS_DIR),
        '--nnunet-root', str(LOCAL_NNUNET_ROOT),
        '--dataset-id', str(DATASET_ID),
        '--workers', '2',
    ]
    subprocess.run(prepare_command, cwd=PROJECT_ROOT, check=True)

if needs_preprocess:
    preprocess_command = [
        'nnUNetv2_plan_and_preprocess',
        '-d', str(DATASET_ID),
        '--verify_dataset_integrity',
        '-c', CONFIGURATION,
        '-pl', PLANNER,
        '-np', str(PREPROCESS_PROCESSES),
    ]
    print('开始 fingerprint、planning 和 preprocessing……')
    subprocess.run(preprocess_command, cwd=PROJECT_ROOT, check=True)

    print('正在将预处理结果备份到 Drive，供下次 Colab 会话恢复……')
    shutil.copytree(local_preprocessed, PERSIST_PREPROCESSED, dirs_exist_ok=True)
else:
    print('预处理缓存已恢复，跳过 planning 和 preprocessing。')

if not plans_path.is_file():
    raise RuntimeError(f'预处理完成后仍未找到 plans：{plans_path}')
print('数据准备完成：', plans_path)

## 4. GPU 训练（支持自动续训）

nnU-Net 默认每 50 epochs 保存一次 checkpoint。Drive 中已有 `checkpoint_latest.pth` 时，本单元自动加入 `--continue-training`；已有 `checkpoint_final.pth` 时不再重复训练。训练过程中可以在 Drive 的 fold 目录查看 `progress.png`。

In [ ]:
model_root = (
    PERSIST_RESULTS
    / 'Dataset137_BraTS2021'
    / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
)
fold_dir = model_root / f'fold_{FOLD}'
latest_checkpoint = fold_dir / 'checkpoint_latest.pth'
final_checkpoint = fold_dir / 'checkpoint_final.pth'

if final_checkpoint.is_file():
    print('最终模型已存在，跳过训练：', final_checkpoint)
else:
    train_command = [
        sys.executable, '-m', 'segmentation.train',
        '--nnunet-root', str(LOCAL_NNUNET_ROOT),
        '--dataset-id', str(DATASET_ID),
        '--configuration', CONFIGURATION,
        '--folds', FOLD,
        '--trainer', TRAINER,
        '--plans', PLANS,
        '--device', 'cuda',
        '--gpu-ids', '0',
        '--num-gpus', '1',
        '--npz',
    ]
    if latest_checkpoint.is_file():
        train_command.append('--continue-training')
        print('发现 checkpoint，开始断点续训：', latest_checkpoint)
    else:
        print('开始新的训练：', TRAINER, 'fold', FOLD)
    subprocess.run(train_command, cwd=PROJECT_ROOT, check=True)

if not final_checkpoint.is_file():
    raise RuntimeError(
        '本次会话未生成 checkpoint_final.pth；'
        '重新连接 Colab 后从头运行 Notebook 即可续训。'
    )
print('训练完成：', final_checkpoint)

## 5. 验证结果和导出模型

训练正常结束后 nnU-Net 会自动进行验证。下面确认 `summary.json`，必要时单独补跑验证，并把 Dataset137 模型目录打包到 Drive。

In [ ]:
import json

validation_summary = fold_dir / 'validation' / 'summary.json'
if not validation_summary.is_file():
    print('未发现验证汇总，开始补跑验证……')
    validate_command = [
        sys.executable, '-m', 'segmentation.train',
        '--nnunet-root', str(LOCAL_NNUNET_ROOT),
        '--dataset-id', str(DATASET_ID),
        '--configuration', CONFIGURATION,
        '--folds', FOLD,
        '--trainer', TRAINER,
        '--plans', PLANS,
        '--device', 'cuda',
        '--gpu-ids', '0',
        '--validation-only',
        '--npz',
    ]
    subprocess.run(validate_command, cwd=PROJECT_ROOT, check=True)

if validation_summary.is_file():
    metrics = json.loads(validation_summary.read_text(encoding='utf-8'))
    print(json.dumps(metrics.get('foreground_mean', metrics.get('mean', metrics)), indent=2)[:4000])
else:
    print('警告：未找到 validation/summary.json，请检查上方验证日志。')

In [ ]:
EXPORT_DIR = DRIVE_PROJECT_DIR / 'exported_models'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
archive_base = EXPORT_DIR / f'Dataset137_BraTS2021_{TRAINER}_fold{FOLD}'
archive_path = Path(shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=PERSIST_RESULTS,
    base_dir='Dataset137_BraTS2021',
))

print('模型已导出：', archive_path)
print('本机后端配置：')
print(f'  BTA_NNUNET_TRAINER={TRAINER}')
print(f'  BTA_NNUNET_FOLDS=["{FOLD}"]')
print('  BTA_NNUNET_PLANS=nnUNetResEncUNetMPlans')
print('  BTA_NNUNET_CONFIGURATION=3d_fullres')
print('  BTA_NNUNET_CHECKPOINT=checkpoint_final.pth')

## 6. 完整实验如何继续

- 实习演示：保留 `TRAINING_PROFILE='demo_100epochs'` 和 `FOLD='0'`。后端也必须配置同一个 trainer 和 fold。
- 标准实验：改为 `TRAINING_PROFILE='full_1000epochs'`，依次训练 folds 0～4；每个 fold 可跨多个 Colab 会话断点续训。
- 只完成一个 fold 时，不要在论文或答辩中声称完成了五折交叉验证，也不要虚构 Dice。以 `validation/summary.json` 中的真实结果为准。
- Colab 免费 GPU 的型号、时长和可用性不保证。长训练中断是正常情况，重新运行 Notebook 会恢复本地预处理缓存并续训。
- 训练权重仅用于研究演示，不能作为疾病确诊依据。